In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix

from datasets import load_dataset, Dataset
from collections import Counter
import numpy as np
import random
#import evaluate

from transformers import BertConfig, BertModel, BertTokenizerFast, BertForSequenceClassification, TrainingArguments, Trainer, get_linear_schedule_with_warmup, get_cosine_schedule_with_warmup

In [5]:
def set_random_seed(seed):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # <- add this line
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

set_random_seed(224)

In [3]:
dataset = load_dataset("k1tub/sentiment_dataset")
dataset = dataset['train']

In [8]:
label_datasets = {}
for label in set(dataset['label']):
    label_datasets[label] = dataset.filter(lambda x: x['label'] == label)

num_classes = len(label_datasets)
samples_per_class = 10_000 // num_classes

balanced_samples = []
for label, subset in label_datasets.items():
    subset = subset.shuffle()
    balanced_samples.append(subset.select(range(min(samples_per_class, len(subset)))))

balanced_dataset = Dataset.from_dict({
    key: sum([ds[key] for ds in balanced_samples], [])
    for key in dataset.column_names
}).shuffle()

In [13]:
ds_train = balanced_dataset[0:8000]
ds_val = balanced_dataset[8000:9000]
ds_test = balanced_dataset[9000:10000]

In [12]:
tokenizer = BertTokenizerFast.from_pretrained('DeepPavlov/rubert-base-cased')

In [42]:
def slice_token(index, sentences, labels, tokenizer, max_length):
    start, stop, step = index.indices(len(sentences))
    result = []
    for i in range(start, stop, step):
        encoding = tokenizer(
                sentences[i],
                padding='max_length',
                truncation = True,
                max_length = max_length,
                return_tensors = 'pt'
            )
        encoding['labels'] = [labels[i]]
        result.append({key : value[0] for key, value in encoding.items()})

    return result

In [43]:
class NERDataset(Dataset):
    def __init__(self, sentences, labels, tokenizer, max_length):
        self.sentences = sentences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        if isinstance(idx, slice):
            # Обработка среза
            return slice_token(idx, self.sentences, self.labels, self.tokenizer, self.max_length)
        elif isinstance(idx, int):
            tokens = self.sentences[idx]
            tag = self.labels[idx]

            # токенизируем
            encoding = self.tokenizer(
                tokens,
                padding='max_length',
                truncation = True,
                max_length = self.max_length,
                return_tensors = 'pt'
            )
            encoding['labels'] = torch.tensor([tag])
            encoding['input_ids'] = torch.squeeze(encoding["input_ids"], 0)
            encoding["token_type_ids"] = torch.squeeze(encoding["token_type_ids"], 0)
            encoding["attention_mask"] = torch.squeeze(encoding["attention_mask"], 0)

            return encoding
            #return {key : value[0] for key, value in encoding.items()}

In [44]:
dataset_train = NERDataset(ds_train['text'], ds_train['label'], tokenizer, 256)
dataset_test = NERDataset(ds_test['text'], ds_test['label'], tokenizer, 256)
dataset_val = NERDataset(ds_val['text'], ds_val['label'], tokenizer, 256)

In [48]:
little_train_loader = DataLoader(dataset_train[0:3000], batch_size=16)
little_val_loader = DataLoader(dataset_val[0:500], batch_size = 16)

In [ ]:
test_loader = DataLoader(dataset_test, batch_size=32, pin_memory=True)
train_loader = DataLoader(dataset_train, batch_size=32)
val_loader = DataLoader(dataset_val, batch_size = 32)

In [ ]:
model = BertForSequenceClassification.from_pretrained('DeepPavlov/rubert-base-cased', num_labels = 3)

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
torch.cuda.empty_cache()

In [ ]:
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), correct_bias=False) # а здесь можно не фильтровать?
total_steps = len(dataset_train) * 5 * 32

scheduler = get_cosine_schedule_with_warmup(
  optimizer,
  num_warmup_steps=total_steps*0.05,
  num_training_steps=total_steps
)

/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [ ]:
training_args = TrainingArguments(
    output_dir='./results',          # Output directory
    eval_strategy="epoch",    # Evaluate after each epoch
    learning_rate=2e-5,             # Learning rate
    per_device_train_batch_size=16, # Batch size for training
    per_device_eval_batch_size=16,  # Batch size for evaluation
    num_train_epochs=2,             # Number of epochs
    weight_decay=0.01,              # Strength of weight decay
    logging_dir="./logs",           # Directory for storing logs
    logging_steps=10,               # что это такое?
    save_strategy="epoch",          # Save model after each epoch
    load_best_model_at_end=True,    # Load the best model after training
    metric_for_best_model="f1", # Use F1 score to choose the best model
)

In [ ]:
def compute_metrics(p):

    predictions, labels = p
    # логиты в индексы
    predictions = predictions.argmax(axis=-1)
    cm = confusion_matrix(labels, predictions)

    print("Confusion Matrix:\n", cm)
    # пихнем в метрику и получим результат
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:
def hp_space_fn(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-3,log=True),
        "weight_decay" : trial.suggest_float("weight_decay", 1e-5, 0.1, log=True) # логарифмический масштаб для lr
        }

def model_init():
    return BertForSequenceClassification.from_pretrained('google-bert/bert-base-uncased', num_labels = len(set(ds['train']['label'])))

In [ ]:
trainer = Trainer(
    model=model,
    #model_init = model_init,
    args=training_args,
    train_dataset=little_train_loader.dataset,  # Training dataset
    eval_dataset=little_val_loader.dataset,   # Evaluation dataset
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    #optimizers=(optimizer, scheduler)
    )


In [ ]:
# собственно обучение - автоматически делает логи
trainer.train()

# оценим модельку
eval_results = trainer.evaluate()
print(f"Evaluation Results: {eval_results}")

# Сохраним, что получилось
trainer.save_model("./sentiment_model")
#b5a31e3a762dc4fdbd905c7a205899ee8116917a

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: lizaolva123 (lizaolva123-rggu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.824800,0.743967,0.676000,0.658538,0.664874,0.671557
2,0.772700,0.727989,0.669333,0.669015,0.673579,0.667392
3,0.576000,0.726751,0.686667,0.677361,0.676246,0.683235


Confusion Matrix:
 [[228  21   6]
 [ 79  99  64]
 [ 24  49 180]]
Confusion Matrix:
 [[179  67   9]
 [ 40 131  71]
 [  5  56 192]]
Confusion Matrix:
 [[199  49   7]
 [ 53 113  76]
 [ 10  40 203]]


Confusion Matrix:
 [[199  49   7]
 [ 53 113  76]
 [ 10  40 203]]
Evaluation Results: {'eval_loss': 0.7267507910728455, 'eval_accuracy': 0.6866666666666666, 'eval_f1': 0.6773605603392837, 'eval_precision': 0.6762460450390312, 'eval_recall': 0.6832352823750174, 'eval_runtime': 24.0321, 'eval_samples_per_second': 31.208, 'eval_steps_per_second': 0.999, 'epoch': 3.0}


In [ ]:
def eval_model(model, data_loader, device):
  model = model.eval()

  all_preds = torch.tensor([], device=device)
  all_trues = torch.tensor([], device=device)

  with torch.no_grad():
    for d in data_loader:
      input_ids = d["input_ids"].to(device)
      attention_mask = d["attention_mask"].to(device)
      targets = d["labels"].to(device)

      outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
      )
      #print(outputs)
      preds = torch.argmax(outputs['logits'], axis=-1)
      all_preds = torch.cat((all_preds, preds), -1)
      all_trues = torch.cat((all_trues, targets), -1)

  precision, recall, f1, _ = precision_recall_fscore_support(all_trues.cpu(), all_preds.cpu(), average='macro')
  acc = accuracy_score(all_trues.cpu(), all_preds.cpu())
  return {
      'accuracy': acc,
      'f1': f1,
      'precision': precision,
      'recall': recall
  }

In [ ]:
test = eval_model(model, test_loader, device)

In [ ]:
test

{'accuracy': 0.498,
 'f1': 0.49186082461483105,
 'precision': 0.4945688694129702,
 'recall': 0.498}

In [ ]:
precision, recall, f1, _ = precision_recall_fscore_support(torch.tensor(ds['test']['label']), torch.zeros(1500), average='macro')
acc = accuracy_score(torch.tensor(ds['test']['label']), torch.zeros(1500))
print({
      'accuracy': acc,
      'f1': f1,
      'precision': precision,
      'recall': recall})

{'accuracy': 0.3333333333333333, 'f1': 0.16666666666666666, 'precision': 0.1111111111111111, 'recall': 0.3333333333333333}


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True) # Подключите Google Drive
!cp -r /content/results /content/drive/MyDrive/чекпоинты # Замените на ваши пути

Mounted at /content/drive
